# Dự án Dịch Anh - Việt với GRU Encoder-Decoder

Notebook này triển khai mô hình Seq2Seq sử dụng GRU và so sánh 3 chiến lược giải mã: **Greedy Search**, **Beam Search**, và **Sampling**.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
import pandas as pd
import numpy as np

# Thiết lập thiết bị
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 1. Định nghĩa Kiến trúc Mô hình (GRU)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(embedded)
        return hidden

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

## 2. Tiền xử lý dữ liệu
Sử dụng dataset từ Kaggle: https://www.kaggle.com/datasets/hungnm/englishvietnamese-translation

In [ ]:
def load_data(en_path, vi_path):
    # Hàm load dữ liệu từ file văn bản
    try:
        with open(en_path, 'r', encoding='utf-8') as f: en_data = f.read().splitlines()
        with open(vi_path, 'r', encoding='utf-8') as f: vi_data = f.read().splitlines()
        return en_data, vi_data
    except FileNotFoundError:
        return ["Example sentence"], ["Câu ví dụ"]

# Khởi tạo vocab giả định (Cần thay thế bằng quá trình Tokenization thực tế)
vocab = {'<sos>': 0, '<eos>': 1, '<pad>': 2}

## 3. Triển khai 3 thuật toán Decoding

In [ ]:
def greedy_decode(model, src, max_len, trg_vocab):
    model.eval()
    with torch.no_grad():
        hidden = model.encoder(src)
    trg_indexes = [trg_vocab['<sos>']]
    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(src.device)
        with torch.no_grad():
            output, hidden = model.decoder(trg_tensor, hidden)
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)
        if pred_token == trg_vocab['<eos>']: break
    return trg_indexes

def sample_decode(model, src, max_len, trg_vocab, temperature=1.0):
    model.eval()
    with torch.no_grad():
        hidden = model.encoder(src)
    trg_indexes = [trg_vocab['<sos>']]
    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(src.device)
        with torch.no_grad():
            output, hidden = model.decoder(trg_tensor, hidden)
        probs = F.softmax(output / temperature, dim=1)
        pred_token = torch.multinomial(probs, 1).item()
        trg_indexes.append(pred_token)
        if pred_token == trg_vocab['<eos>']: break
    return trg_indexes

def beam_search_decode(model, src, max_len, trg_vocab, beam_size=3):
    model.eval()
    with torch.no_grad():
        hidden = model.encoder(src)
    beams = [(0, [trg_vocab['<sos>']], hidden)]
    for i in range(max_len):
        candidates = []
        for score, indices, h in beams:
            if indices[-1] == trg_vocab['<eos>']:
                candidates.append((score, indices, h))
                continue
            trg_tensor = torch.LongTensor([indices[-1]]).to(src.device)
            with torch.no_grad():
                output, new_h = model.decoder(trg_tensor, h)
            log_probs = F.log_softmax(output, dim=1)
            topk_probs, topk_idx = log_probs.topk(beam_size)
            for j in range(beam_size):
                candidates.append((score + topk_probs[0][j].item(), indices + [topk_idx[0][j].item()], new_h))
        beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_size]
        if all(b[1][-1] == trg_vocab['<eos>'] for b in beams): break
    return beams[0][1]

## 4. Huấn luyện và So sánh

In [ ]:
def compare_strategies(model, src_tensor, trg_vocab):
    g_res = greedy_decode(model, src_tensor, 50, trg_vocab)
    s_res = sample_decode(model, src_tensor, 50, trg_vocab)
    b_res = beam_search_decode(model, src_tensor, 50, trg_vocab, beam_size=3)

    print(f"Greedy Results: {g_res}")
    print(f"Sampling Results: {s_res}")
    print(f"Beam Search Results: {b_res}")

In [ ]:
# Tham số giả định cho việc khởi tạo mô hình
INPUT_DIM = 100
OUTPUT_DIM = 100
EMB_DIM = 32
HID_DIM = 64
N_LAYERS = 2
DROPOUT = 0.5

# Khởi tạo mô hình
enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

# Tạo một src_tensor giả lập (ví dụ: một câu có 5 tokens)
src_tensor = torch.randint(0, INPUT_DIM, (5, 1)).to(device)

print("--- So sánh các chiến lược Decoding ---")
compare_strategies(model, src_tensor, vocab)

--- So sánh các chiến lược Decoding ---
Greedy: [0, 11, 11, 11, 94, 94, 20, 20, 20, 20, 20, 98, 20, 20, 20, 98, 20, 20, 20, 98, 20, 20, 20, 20, 98, 20, 20, 20, 98, 20, 20, 20, 20, 98, 20, 20, 20, 98, 20, 20, 20, 20, 98, 20, 20, 20, 98, 20, 20, 20, 20]
Sampling: [0, 9, 84, 87, 51, 79, 54, 25, 90, 50, 15, 92, 87, 85, 33, 95, 77, 90, 67, 50, 21, 3, 2, 35, 84, 29, 83, 3, 15, 21, 45, 65, 72, 51, 53, 45, 3, 36, 49, 13, 42, 57, 3, 32, 11, 93, 63, 43, 45, 89, 54]
Beam Search: [0, 11, 11, 11, 11, 94, 94, 20, 20, 20, 20, 20, 98, 20, 20, 20, 98, 20, 20, 20, 20, 98, 20, 20, 20, 98, 20, 20, 20, 20, 98, 20, 20, 20, 98, 20, 20, 20, 20, 98, 20, 20, 20, 98, 20, 20, 20, 20, 98, 20, 20]


Dưới đây là hướng dẫn xây dựng mô hình **Encoder-Decoder** sử dụng kiến trúc **GRU** (thường được ưu tiên vì hiệu năng tốt và huấn luyện nhanh hơn LSTM) cho bài toán dịch Anh - Việt, cùng với việc triển khai 3 chiến lược giải mã (Decoding Strategies).

---

## 1. Kiến trúc Mô hình (GRU Encoder-Decoder)

Chúng ta sẽ sử dụng kiến trúc sequence-to-sequence (Seq2Seq) cơ bản:

* **Encoder:** Chuyển câu tiếng Anh thành một vector ngữ cảnh (context vector).
* **Decoder:** Nhận vector ngữ cảnh và dự đoán từng từ tiếng Việt tương ứng.

```python
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        # src: [src_len, batch_size]
        embedded = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(embedded)
        # hidden: [n_layers, batch_size, hid_dim]
        return hidden

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden):
        # input: [batch_size]
        # hidden: [n_layers, batch_size, hid_dim]
        input = input.unsqueeze(0) # [1, batch_size]
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden

```

---

## 2. Triển khai 3 thuật toán Decoding

Sau khi mô hình đã được huấn luyện, bước Inference (suy luận) sẽ sử dụng các chiến lược sau:

### A. Greedy Search

Lấy từ có xác suất cao nhất tại mỗi bước.

* **Đặc điểm:** Nhanh nhất, nhưng dễ bị lặp từ hoặc đi vào "ngõ cụt" nếu từ đầu tiên chọn sai.

### B. Beam Search

Giữ lại $k$ ứng cử viên tốt nhất (beam width) ở mỗi bước thay vì chỉ 1.

* **Đặc điểm:** Cho kết quả chính xác nhất, câu văn trôi chảy hơn nhưng tốn tài nguyên tính toán hơn.

### C. Sampling

Chọn từ ngẫu nhiên dựa trên phân phối xác suất (thường dùng kèm `Temperature`).

* **Đặc điểm:** Tạo ra kết quả đa dạng, sáng tạo nhưng đôi khi thiếu logic hoặc sai ngữ pháp.

---

## 3. So sánh và Đánh giá

Dựa trên yêu cầu của bạn về việc **"cái nào làm nhanh nhất và dễ nhất"**:

| Tiêu chí | Greedy Search | Beam Search | Sampling |
| --- | --- | --- | --- |
| **Độ khó cài đặt** | **Dễ nhất** (Chỉ cần `argmax`) | Khó nhất (Cần quản lý danh sách $k$ ứng cử viên) | Trung bình (Sử dụng `torch.multinomial`) |
| **Tốc độ xử lý** | **Nhanh nhất** | Chậm nhất (Tỉ lệ thuận với Beam Width) | Nhanh |
| **Chất lượng dịch** | Trung bình | **Tốt nhất** | Kém ổn định nhất |

### Kết luận:

Nếu mục tiêu của bạn là **nhanh nhất và dễ nhất**, hãy chọn **Greedy Search**.

**Lý do:**

1. **Code tối giản:** Bạn chỉ cần một vòng lặp `for` và lấy vị trí có giá trị lớn nhất từ output của Decoder.
2. **Không tham số:** Bạn không cần tinh chỉnh `beam_size` hay `temperature`.
3. **Phù hợp Starter:** Với dataset từ Kaggle bạn cung cấp, Greedy Search đủ để kiểm tra xem mô hình của bạn có đang học đúng hướng hay không trước khi nâng cấp lên các kỹ thuật phức tạp hơn.

> **Gợi ý thêm:** Khi xử lý tiếng Việt, bạn nên chú ý bước tiền xử lý (Tokenization). Vì tiếng Việt là ngôn ngữ đơn lập, bạn có thể dùng thư viện `pyvi` hoặc `underthesea` để tách từ (word segmentation) trước khi đưa vào mô hình để đạt độ chính xác cao hơn.

In [ ]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src: [src_len, batch_size]
        embedded = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(embedded)
        # hidden: [n_layers, batch_size, hid_dim]
        return hidden

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden):
        # input: [batch_size]
        # hidden: [n_layers, batch_size, hid_dim]
        input = input.unsqueeze(0) # [1, batch_size]
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden

# 2. Tải và Tiền xử lý dữ liệu
Chúng ta sẽ sử dụng thư viện `torchtext` và `spacy` (hoặc `pyvi` cho tiếng Việt) để xử lý dữ liệu.

In [ ]:
import pandas as pd
import numpy as np
# Giả sử bạn đã tải file từ Kaggle lên Colab
# en_sents.txt và vi_sents.txt

def load_data(en_path, vi_path):
    with open(en_path, 'r', encoding='utf-8') as f:
        en_data = f.read().splitlines()
    with open(vi_path, 'r', encoding='utf-8') as f:
        vi_data = f.read().splitlines()
    return en_data, vi_data

# Placeholder cho đường dẫn (Thay đổi theo thực tế)
# en_data, vi_data = load_data('en_sents.txt', 'vi_sents.txt')
print("Data loading functions ready.")

Data loading functions ready.


# 3. Triển khai các thuật toán Decoding

In [ ]:
import random
import torch.nn.functional as F

def greedy_decode(model, src, max_len, trg_vocab):
    model.eval()
    with torch.no_grad():
        hidden = model.encoder(src)

    # <sos> token
    trg_indexes = [trg_vocab['<sos>']]

    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(src.device)
        with torch.no_grad():
            output, hidden = model.decoder(trg_tensor, hidden)

        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)

        if pred_token == trg_vocab['<eos>']:
            break

    return trg_indexes

def sample_decode(model, src, max_len, trg_vocab, temperature=1.0):
    model.eval()
    with torch.no_grad():
        hidden = model.encoder(src)

    trg_indexes = [trg_vocab['<sos>']]

    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(src.device)
        with torch.no_grad():
            output, hidden = model.decoder(trg_tensor, hidden)

        probs = F.softmax(output / temperature, dim=1)
        pred_token = torch.multinomial(probs, 1).item()
        trg_indexes.append(pred_token)

        if pred_token == trg_vocab['<eos>']:
            break

    return trg_indexes

def beam_search_decode(model, src, max_len, trg_vocab, beam_size=3):
    # Simplified Beam Search implementation
    model.eval()
    with torch.no_grad():
        hidden = model.encoder(src)

    # Each element: (score, [indices], hidden_state)
    beams = [(0, [trg_vocab['<sos>']], hidden)]

    for i in range(max_len):
        candidates = []
        for score, indices, h in beams:
            if indices[-1] == trg_vocab['<eos>']:
                candidates.append((score, indices, h))
                continue

            trg_tensor = torch.LongTensor([indices[-1]]).to(src.device)
            with torch.no_grad():
                output, new_h = model.decoder(trg_tensor, h)

            log_probs = F.log_softmax(output, dim=1)
            topk_probs, topk_idx = log_probs.topk(beam_size)

            for j in range(beam_size):
                candidates.append((score + topk_probs[0][j].item(), indices + [topk_idx[0][j].item()], new_h))

        beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_size]
        if all(b[1][-1] == trg_vocab['<eos>'] for b in beams): break

    return beams[0][1]

# 4. Huấn luyện mô hình và So sánh kết quả

In [ ]:
# Giả lập hàm so sánh kết quả dịch
def compare_strategies(model, src_sentence, trg_vocab):
    # Chuyển src_sentence sang tensor...
    # src_tensor = ...

    greedy_res = greedy_decode(model, src_tensor, 50, trg_vocab)
    sample_res = sample_decode(model, src_tensor, 50, trg_vocab)
    beam_res = beam_search_decode(model, src_tensor, 50, trg_vocab)

    print("Greedy:", greedy_res)
    print("Sampling:", sample_res)
    print("Beam Search:", beam_res)

# 5. Thiết lập Huấn luyện

Phần này khởi tạo các thành phần cần thiết và chạy vòng lặp huấn luyện.

In [ ]:
# Định nghĩa Hyperparameters
LEARNING_RATE = 0.001
N_EPOCHS = 10
CLIP = 1

# Khởi tạo Criterion (bỏ qua index của <pad> token)
TRG_PAD_IDX = vocab.get('<pad>', 2)
criterion = nn.CrossEntropyLoss(ignore_index=TRG_PAD_IDX)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0

    for i, batch in enumerate(iterator):
        src = batch['src'].to(device)
        trg = batch['trg'].to(device)

        optimizer.zero_grad()

        # Với Seq2Seq cơ bản, chúng ta cần thực hiện forward pass thủ công hoặc cập nhật class Seq2Seq
        # Ở đây tôi giả định một bước huấn luyện đơn giản cho Encoder-Decoder
        encoder_hidden = model.encoder(src)

        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = model.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(device)
        hidden = encoder_hidden
        input = trg[0,:]

        for t in range(1, trg_len):
            output, hidden = model.decoder(input, hidden)
            outputs[t] = output
            teacher_force = random.random() < 0.5
            top1 = output.argmax(1)
            input = trg[t] if teacher_force else top1

        outputs = outputs[1:].view(-1, trg_vocab_size)
        trg = trg[1:].view(-1)

        loss = criterion(outputs, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

# Giả lập dummy data để chạy test
dummy_batch = {
    'src': torch.randint(0, INPUT_DIM, (10, 4)).to(device),
    'trg': torch.randint(0, OUTPUT_DIM, (12, 4)).to(device)
}
dummy_iterator = [dummy_batch] * 5

print("Bắt đầu huấn luyện thử nghiệm...")
for epoch in range(N_EPOCHS):
    train_loss = train(model, dummy_iterator, optimizer, criterion, CLIP)
    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f}')

print("--- So sánh sau khi train thử ---")
compare_strategies(model, src_tensor, vocab)

Bắt đầu huấn luyện thử nghiệm...
Epoch: 01 | Train Loss: 4.572
Epoch: 02 | Train Loss: 4.501
Epoch: 03 | Train Loss: 4.407
Epoch: 04 | Train Loss: 4.281
Epoch: 05 | Train Loss: 4.122
Epoch: 06 | Train Loss: 3.862
Epoch: 07 | Train Loss: 3.573
Epoch: 08 | Train Loss: 3.336
Epoch: 09 | Train Loss: 3.157
Epoch: 10 | Train Loss: 2.988
--- So sánh sau khi train thử ---
Greedy: [0, 18, 18, 18, 18, 69, 17, 88, 88, 88, 88, 88, 88, 88, 26, 88, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26]
Sampling: [0, 37, 18, 30, 51, 78, 41, 3, 52, 50, 70, 10, 3, 17, 47, 50, 18, 10, 64, 7, 71, 79, 28, 70, 70, 88, 3, 41, 26, 26, 50, 12, 32, 52, 26, 47, 51, 88, 64, 70, 46, 17, 10, 26, 26, 4, 17, 28, 51, 50, 78]
Beam Search: [0, 18, 18, 18, 18, 45, 17, 17, 88, 88, 26, 88, 88, 26, 88, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26,